# Predicting Weather with Machine Learning

This notebook demonstrates how to predict temperatures using historical data from `Temperatures.csv`.

## 1. Loading the Data

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_absolute_error

df = pd.read_csv('Temperatures.csv')
df.head()

## 2. Data Exploration

In [ ]:
print("Shape of the dataset:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nInfo:")
df.info()
print("\nNull values:")
print(df.isnull().sum())

## 3. Data Preparation (Wrangling)

In [ ]:
# Drop high cardinality uncertainty columns
cols_to_drop = [col for col in df.columns if 'Uncertainty' in col]
df.drop(columns=cols_to_drop, inplace=True)

# Convert temperatures from Celsius to Fahrenheit
def c_to_f(celsius_temp):
    return (celsius_temp * 9/5) + 32

temp_cols = [col for col in df.columns if 'Temperature' in col]
for col in temp_cols:
    df[col] = df[col].apply(c_to_f)

# Convert dt column to DateTime and extract Year and Month
df['dt'] = pd.to_datetime(df['dt'])
df['Year'] = df['dt'].dt.year
df['Month'] = df['dt'].dt.month

# Filter data from the year 1850 onwards
df = df[df['Year'] >= 1850]

# Drop N/A values
df.dropna(inplace=True)

df.head()

## 4. Visualization

In [ ]:
plt.figure(figsize=(10, 8))
sns.heatmap(df.corr(numeric_only=True), annot=True, cmap='coolwarm', fmt=".2f")
plt.title('Correlation Heatmap')
plt.show()

## 5. Feature Selection
We will use `LandAndOceanAverageTemperature` as our target, and other temperatures as features.

In [ ]:
target = 'LandAndOceanAverageTemperature'
y = df[target]
X = df[['LandAverageTemperature', 'LandMaxTemperature', 'LandMinTemperature']]
print(X.head())

## 6. Train-Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42)
print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)

## 7. Baseline Evaluation
Calculating Baseline Mean Absolute Error.

In [ ]:
baseline_pred = [y_train.mean()] * len(y_train)
baseline_mae = mean_absolute_error(y_train, baseline_pred)
print("Baseline MAE:", baseline_mae)

## 8. Model Training

In [ ]:
model = make_pipeline(
    RandomForestRegressor(random_state=42)
)
model.fit(X_train, y_train)

## 9. Model Evaluation
Calculate Mean Absolute Percentage Error (MAPE).

In [ ]:
# Make predictions
predictions = model.predict(X_test)

# Calculate MAE and MAPE
mae = mean_absolute_error(y_test, predictions)
errors = abs(predictions - y_test)
mape = 100 * (errors / y_test)
accuracy = 100 - np.mean(mape)

print("Mean Absolute Error:", round(mae, 2), "degrees.")
print("Model Accuracy:", round(accuracy, 2), "%.")